# 01 · Del texto plano al texto anotado

## Propósito

Entender qué le agrega spaCy a un texto que Python solo no puede ver, y por qué
`texto.split()` no alcanza.

Al final vas a poder leer un pipeline de spaCy, saber qué hace cada componente y
manejar los tres objetos con los que vamos a trabajar toda la unidad: `Doc`,
`Token` y `Span`.

## Qué consume

- `datos/corpus.jsonl`

## Resultados de aprendizaje

- Distinguir un token de una palabra.
- Leer `nlp.pipe_names` y decir qué hace cada componente.
- Explicar por qué la segmentación en oraciones no es "cortar en los puntos".

## ◈ Microglosario

Cuatro palabras que vas a usar toda la unidad. No hace falta memorizarlas ahora: están
acá para volver.

| Término | Qué es |
|---|---|
| **pipeline** | La fila de componentes que spaCy le pasa a cada texto, uno detrás de otro |
| **`Doc`** | Un texto ya procesado. No es una cadena: es una cadena más todo lo que el pipeline le anotó encima |
| **`Token`** | Cada unidad mínima del `Doc`. Casi siempre una palabra, a veces un signo |
| **`Span`** | Un pedazo de `Doc`: uno o varios tokens seguidos. Una entidad y un grupo nominal son `Span` |
| **tokenizar** | Cortar el texto en tokens. Es lo primero que pasa, antes que cualquier análisis |

## 1. Preparación

In [ ]:
# spacy: la librería de análisis lingüístico.
import spacy

# pandas: para trabajar el corpus como una tabla.
import pandas as pd

# es_core_news_lg: modelo en español (es), entrenado con noticias (news), grande (lg)
nlp = spacy.load("es_core_news_lg")

# lines=True: el archivo .jsonl tiene un registro JSON por renglón
corpus = pd.read_json("datos/corpus.jsonl", lines=True)

cantidad_parrafos = len(corpus)
# :, separa los miles con una coma (5,428 en vez de 5428)
print("párrafos en el corpus:", f"{cantidad_parrafos:,}")

# pipe_names: la lista de componentes que tiene el pipeline cargado
if "ner" not in nlp.pipe_names:
    print("ATENCIÓN: el modelo se cargó sin NER. Revisá que sea es_core_news_lg y no otro.")

## 2. El problema: para Python, un texto es una fila de caracteres

Tomemos un párrafo del corpus.

In [ ]:
# .loc[1, "texto"] devuelve el texto de la fila 1.
parrafo = corpus.loc[1, "texto"]

print(parrafo)

Para Python esto es un `str`: una fila de caracteres, nada más. No sabe
dónde termina una palabra, cuál es un verbo, ni que *Mercedes* es un lugar.

La forma más obvia de separarlo en palabras es cortar por los espacios.

In [ ]:
# .split() sin argumentos corta el texto en cada espacio en blanco.
palabras = parrafo.split()
cantidad_de_piezas = len(palabras)

print("piezas obtenidas:", cantidad_de_piezas)
print()
# [:15]: los primeros 15 elementos de la lista
print(palabras[:15])

Funciona, hasta que no funciona. Miremos qué pasa con dos casos que están
por todo nuestro corpus.

In [ ]:
# Este ejemplo tiene dos problemas típicos del corpus:
#   1) un punto pegado a la palabra siguiente (herencia de la reconstrucción)
#   2) un verbo con pronombre enclítico ("contame")
ejemplo = "Fui al kiosco de la esquina.La vecina me saludó y contame vos qué hago."

piezas_del_ejemplo = ejemplo.split()
print(piezas_del_ejemplo)

`'esquina.La'` quedó como **una sola pieza**. Si contás palabras, esa
cuenta como una palabra que no existe. Y como aparece 5.416 veces en el corpus,
no es un caso raro: es sistemático.

Veamos qué hace spaCy con lo mismo.

## 3. El pipeline

Antes de procesar nada, miremos la herramienta. `nlp` no es una función: es una
**cadena de componentes** que corren en orden, y cada uno agrega una capa de
anotación al texto.

In [ ]:
# pipe_names: los componentes del pipeline, en el orden en que corren
print(nlp.pipe_names)

Qué hace cada uno:

| componente | qué agrega |
|---|---|
| `tok2vec` | Convierte cada token en un vector de contexto. Es la base de la que se alimentan los demás. |
| `morphologizer` | Categoría gramatical (`pos_`) y rasgos morfológicos (`morph`). |
| `parser` | Relaciones sintácticas entre tokens. **También** es lo que delimita las oraciones. |
| `attribute_ruler` | Correcciones por reglas sobre lo que vino antes. |
| `lemmatizer` | La forma de diccionario de cada palabra. |
| `ner` | Entidades nombradas: personas, lugares, organizaciones. |

**La tokenización no está en la lista.** Pasa antes que todo esto y no se puede
desactivar: es la puerta de entrada. Todo lo demás opera sobre los tokens que
ella produjo.

## 4. El objeto `Doc`

Pasar un texto por `nlp()` devuelve un `Doc`: el texto más todas las capas de
anotación.

In [ ]:
doc = nlp(ejemplo)

# type(): qué clase de objeto es. __name__: el nombre de esa clase, como texto
tipo_de_objeto = type(doc)
nombre_del_tipo = tipo_de_objeto.__name__
cantidad_de_tokens = len(doc)

print("tipo   :", nombre_del_tipo)
print("tokens :", cantidad_de_tokens)
print()
for token in doc:
    # token.text: el texto del token, tal como aparece en el original
    print(" ", token.text)

Ahora sí: `esquina` y `.` quedaron separados, y `La` volvió a ser una
palabra.

Fijate también que **`al` quedó entero**. spaCy no lo parte en *a + el*. Eso es
una decisión del modelo español, no un error, y conviene saberla antes de contar
preposiciones.

### Cuánto cambia esto a escala

Un caso suelto no convence a nadie. Midamos sobre 300 párrafos del corpus.

In [ ]:
# Tomamos los primeros 300 párrafos.
primeros = corpus.head(300)
muestra = list(primeros["texto"])

# Contamos las piezas que produce .split()
piezas_split = 0
for texto in muestra:
    piezas_del_texto = texto.split()
    piezas_split = piezas_split + len(piezas_del_texto)

# Contamos los tokens que produce spaCy.
# nlp.pipe(): procesa muchos textos en lotes, más rápido que de a uno.
# disable=: apaga estos componentes. Solo queremos tokenizar, que es lo más liviano.
componentes_apagados = ["morphologizer", "parser", "attribute_ruler", "lemmatizer", "ner"]
tokens_spacy = 0
for documento in nlp.pipe(muestra, disable=componentes_apagados):
    tokens_spacy = tokens_spacy + len(documento)

diferencia = tokens_spacy - piezas_split

# :, separa los miles con una coma
print("con .split() :", f"{piezas_split:,}")
print("con spaCy    :", f"{tokens_spacy:,}")
print("diferencia   :", f"{diferencia:,}")

Miles de unidades de diferencia sobre 300 párrafos. Toda cuenta de
frecuencias hecha con `.split()` sobre este corpus arrastra ese error.

## 5. `Token`: la unidad de análisis

Un `Token` no es solo su texto. Es un objeto con muchos atributos, y en los
próximos cuadernos vamos a usar casi todos. Por ahora, tres de posición.

In [ ]:
doc = nlp(parrafo)

# Armamos una fila por token, con los tres datos de posición.
filas = []
for token in doc[:20]:  # doc[:20]: los primeros 20 tokens
    fila = {
        "posicion": token.i,  # i: el lugar del token dentro del documento
        "token": token.text,
        "empieza_en_caracter": token.idx,  # idx: en qué carácter del texto empieza
        "espacio_despues": token.whitespace_,  # whitespace_: el espacio que lo sigue, o "" si no hay
    }
    filas.append(fila)

# pandas dibuja la tabla solo: no hay que alinear nada a mano.
pd.DataFrame(filas)

- `token.i` — la posición del token dentro del `Doc`.
- `token.idx` — la posición del **carácter** donde empieza, en el texto original.
- `token.whitespace_` — el espacio que venía después.

Ese último es lo que permite reconstruir el texto exacto. spaCy no pierde nada:
un `Doc` siempre se puede volver a convertir en el string del que salió.

In [ ]:
# Reconstruimos el texto original a partir de los tokens.
# Cada token se pega con el espacio que tenía después (whitespace_).
reconstruido = ""
for token in doc:
    reconstruido = reconstruido + token.text + token.whitespace_

es_identico = reconstruido == parrafo
print("¿es idéntico al original?", es_identico)

## 6. `Span`: un pedazo del `Doc`

Cortar un `Doc` devuelve un `Span`: una porción continua de tokens. No es un
string, y conserva todas las anotaciones.

In [ ]:
# doc[0:6]: del token 0 al 5 (el 6 no entra). El resultado es un Span.
trozo = doc[0:6]

tipo_de_objeto = type(trozo)
nombre_del_tipo = tipo_de_objeto.__name__  # __name__: el nombre de la clase, como texto

print("tipo  :", nombre_del_tipo)
print("texto :", trozo.text)
# start y end: dónde empieza y dónde termina el trozo, contado en tokens
print("desde el token", trozo.start, "hasta el", trozo.end)

`Doc`, `Token` y `Span` son los tres objetos de toda la unidad. Las entidades
nombradas que vamos a ver en el `03` son `Span`. Los grupos nominales también.

## 7. Oraciones

spaCy no corta las oraciones buscando puntos: usa el analizador sintáctico.
Eso resuelve casos que cortar por puntos arruinaría. Pero no es infalible, y
conviene verlo con un ejemplo que tiene las dos cosas.

In [ ]:
prueba = nlp("El Dr. Pérez llegó a las 15.30 desde EE.UU. La reunión empezó tarde.")

# .sents: las oraciones que detectó el parser, una por una
numero = 0
for oracion in prueba.sents:
    numero = numero + 1
    print(numero, "|", oracion.text)

**Una sola oración.** Cortar por puntos habría dado cinco pedazos, así que
spaCy acertó tres veces: no cortó en `Dr.`, ni en `15.30`, ni adentro de `EE.UU.`

Pero también **falló una vez**: `EE.UU.` y `La reunión empezó tarde` son dos
oraciones distintas y quedaron pegadas. La abreviatura con puntos internos lo
confundió.

Comprobalo sacando `EE.UU.` de la frase.

In [ ]:
sin_abreviatura = nlp("El Dr. Pérez llegó a las 15.30. La reunión empezó tarde.")

# .sents: las oraciones que detectó el parser, una por una
numero = 0
for oracion in sin_abreviatura.sents:
    numero = numero + 1
    print(numero, "|", oracion.text)

Ahora sí son dos.

Esto no es un defecto que haya que esconder: es la primera aparición de la idea
que atraviesa toda la unidad. **El modelo acierta mucho y falla algunas veces, y
la única forma de saber cuándo es mirando la salida.** Nadie te va a avisar.

Fijate además que cada oración es un `Span`, no un string.

In [ ]:
# .sents no es una lista: entrega las oraciones de a una.
# list() las junta en una lista, y así podemos pedir la primera con [0].
oraciones = list(prueba.sents)
primera = oraciones[0]

tipo_de_objeto = type(primera)
nombre_del_tipo = tipo_de_objeto.__name__
cantidad_de_tokens = len(primera)

print("tipo:", nombre_del_tipo)
print("tokens de la primera oración:", cantidad_de_tokens)

## 8. Averiguar cosas sin preguntarle a nadie

Hasta acá te fui diciendo qué atributo mirar. En tu trabajo no va a haber nadie
diciéndotelo, así que conviene aprender a buscarlo.

Hay tres formas, de la más rápida a la más completa.

**Primera: preguntarle al objeto qué sabe hacer.** `dir()` lista todo lo que un
objeto tiene adentro.

In [ ]:
token = doc[0]

# dir() devuelve muchísimo. Nos quedamos con los atributos que empiezan
# con "is_", que son las preguntas de sí o no que se le pueden hacer.
atributos = []
for nombre in dir(token):
    if nombre.startswith("is_"):  # startswith(): ¿el nombre empieza con "is_"?
        atributos.append(nombre)

cantidad_de_atributos = len(atributos)
print("cantidad:", cantidad_de_atributos)
print(atributos)

Dieciocho preguntas distintas, y en esta unidad vamos a usar cinco. Las otras
trece están ahí para cuando las necesites.

**Segunda: leer la documentación desde adentro de Python.** `help()` imprime la
explicación que los autores de la librería escribieron para esa función.

In [ ]:
# help(): muestra la documentación de cualquier función o método
help(doc.similarity)

Fijate qué te dice ese texto, porque es siempre la misma estructura y sirve
para cualquier función de cualquier librería:

- **qué hace** — "Make a semantic similarity estimate"
- **qué recibe** — `other (object)`: con qué se puede comparar
- **qué devuelve** — `RETURNS (float)`: un número, más alto es más parecido
- **dónde seguir leyendo** — el enlace a `spacy.io/api`

Esas cuatro preguntas —qué hace, qué recibe, qué devuelve, dónde leer más— son
las que hay que hacerle a cualquier función nueva.

**Tercera: la documentación del sitio.** <https://spacy.io/api> tiene una página
por cada objeto. La de `Token` lista todos sus atributos con una descripción de
una línea. Tenela abierta mientras trabajás.

## 9. Provocar un error a propósito

Los errores son información, no accidentes. Y la forma más rápida de entender
qué espera una función es darle algo que no espera.

Las dos celdas que siguen fallan. Están para que leas el mensaje.

In [ ]:
# Pedimos el token número 99 de un documento que tiene menos de 99 tokens.
prueba_corta = nlp("Una prueba corta.")

prueba_corta[99]

`IndexError` y, entre corchetes, un código: `[E026]`. spaCy numera sus errores,
y si buscás ese código en internet vas a encontrar la explicación exacta.

El mensaje además te dice el largo real del documento. No hace falta adivinar.

In [ ]:
# Le asignamos un número a un atributo que espera texto.
token = prueba_corta[0]

token.lemma_ = 5

`TypeError: Argument 'lemma_' has incorrect type (expected str, got int)`.

Dice exactamente qué esperaba y qué recibió. Cuando algo falle en tu trabajo,
leé el mensaje entero antes de cambiar el código: en general está escrito ahí
qué hay que corregir.

## 10. Ejercicio

Elegí un párrafo cualquiera del corpus y contestá con código:

1. ¿Cuántas piezas produce `.split()` y cuántos tokens produce spaCy?
2. ¿Cuántas oraciones tiene según spaCy?
3. ¿Cuántos puntos (`.`) hay en el texto? ¿Coincide con la cantidad de oraciones?
   Si no coincide, buscá por qué.

Cambiá el número de la variable `fila` y volvé a correr. **Hacelo por lo menos
con dos párrafos distintos**: probá también con la fila `141`, donde el desajuste
tiene otra causa.

In [ ]:
fila = 101

texto_ejercicio = corpus.loc[fila, "texto"]
print(texto_ejercicio)
print()

# TU CÓDIGO ACÁ:
# 1. Procesá texto_ejercicio con nlp
# 2. Compará len(texto_ejercicio.split()) con len(doc)
# 3. Contá las oraciones con doc.sents
# 4. Contá los puntos con texto_ejercicio.count(".")

In [ ]:
# --- Solución ---

doc_ejercicio = nlp(texto_ejercicio)

piezas_del_texto = texto_ejercicio.split()
lista_de_oraciones = list(doc_ejercicio.sents)  # list(): para poder contarlas con len()

piezas = len(piezas_del_texto)
tokens = len(doc_ejercicio)
oraciones = len(lista_de_oraciones)
puntos = texto_ejercicio.count(".")  # count("."): cuántas veces aparece el punto

print("piezas con .split() :", piezas)
print("tokens con spaCy    :", tokens)
print("oraciones           :", oraciones)
print("puntos en el texto  :", puntos)
print()

if oraciones != puntos:
    print("No coinciden. Las oraciones detectadas fueron:")
    numero = 0
    for oracion in lista_de_oraciones:
        numero = numero + 1
        comienzo_de_la_oracion = oracion.text[:90]  # [:90]: los primeros 90 caracteres
        print(" ", numero, "|", comienzo_de_la_oracion)

> ### Perilla
>
> `nlp` viene con seis componentes prendidos, y cada uno cuesta tiempo. Si lo
> único que necesitás es tokenizar, los otros cinco trabajan para nada.
>
> Corré la celda de abajo y mirá la diferencia. Después pensá esto: la lista de
> `disable` no es una optimización técnica. **Es la declaración de qué le estás
> preguntando al texto.** Volvemos sobre esto en el cuaderno 06.

In [ ]:
# time: para medir cuánto tarda cada cosa.
import time

primeros = corpus.head(300)

muestra = list(primeros["texto"])

# Los componentes que apagamos para quedarnos solo con la tokenización.
componentes_apagados = ["morphologizer", "parser", "attribute_ruler", "lemmatizer", "ner"]

# time.time(): la hora actual en segundos. La resta da cuánto tardó.
inicio = time.time()
for documento in nlp.pipe(muestra):
    pass  # pass: no hacer nada. Solo queremos que spaCy procese
completo = time.time() - inicio

inicio = time.time()
for documento in nlp.pipe(muestra, disable=componentes_apagados):
    pass
solo_tokens = time.time() - inicio

veces_mas_rapido = completo / solo_tokens

# :.1f: un decimal. :.0f: sin decimales
print(f"pipeline completo   : {completo:.1f} s")
print(f"solo tokenizando    : {solo_tokens:.1f} s")
print(f"veces más rápido    : {veces_mas_rapido:.0f}x")

## Lo que hiciste hoy

- Viste por qué `.split()` no alcanza, y cuánto se desvía a escala.
- Leíste el pipeline como un objeto y sabés qué hace cada componente.
- Manejaste `Doc`, `Token` y `Span`.
- Entendiste que las oraciones las delimita el analizador sintáctico, no los puntos.

## Lo que viene

En el `02` vamos a las capas que agregan los componentes: lema, categoría
gramatical y morfología. Ahí aparece lo más rico que tiene el español y lo que
la clase de text mining no podía ver.